In [2]:
import os
import glob
import json
import random
import joblib
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
# DATASET_DIR = r"dataset-IR"
DATASET_DIR = r"E:\MLX90640\Abbas Dataset\dataset-IR"

print(f"Dataset Directory set to: {os.path.abspath(DATASET_DIR)}")


def audit_dataset_integrity(base_dir):
    """
    Audits all subfolders in dataset-IR to check frame count alignment
    between .joblib, .json, and _image directories.
    """
    subfolders = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    subfolders.sort()
    
    summary = []
    for folder in subfolders:
        fpath = os.path.join(base_dir, folder)
        joblib_path = os.path.join(fpath, f"{folder}.joblib")
        json_path = os.path.join(fpath, f"{folder}.json")
        img_dir = os.path.join(fpath, f"{folder}_image")
        
        joblib_count = len(joblib.load(joblib_path)) if os.path.exists(joblib_path) else 0
        img_count = len(glob.glob(os.path.join(img_dir, "*.*"))) if os.path.exists(img_dir) else 0
        
        json_count = 0
        if os.path.exists(json_path):
            with open(json_path, 'r', encoding='utf-8') as f:
                jdata = json.load(f)
                if folder in jdata:
                    json_count = len(jdata[folder])
                    
        summary.append({
            "folder": folder,
            "joblib_frames": joblib_count,
            "json_labels": json_count,
            "image_files": img_count,
            "aligned": (joblib_count == img_count == json_count)
        })
        
    print(f"\n{'Subfolder Name':<32} | {'Joblib':<8} | {'JSON':<8} | {'Images':<8} | {'Aligned?':<8}")
    print("-" * 75)
    for s in summary:
        print(f"{s['folder']:<32} | {s['joblib_frames']:<8} | {s['json_labels']:<8} | {s['image_files']:<8} | {str(s['aligned']):<8}")


def process_thermal_pixels(pixels, sensor_shape=(24, 32), target_shape=(320, 240)):
    """
    Processes raw MLX90640 thermal pixels:
      1. Converts to float numpy array.
      2. Imputes NaNs with local median.
      3. Reshapes to 24x32 grid.
      4. Upscales to 320x240 using cubic interpolation.
    """
    px = np.array(pixels, dtype=float)
    
    # Handle NaN values (defective pixels)
    if np.isnan(px).any():
        median_val = np.nanmedian(px)
        px[np.isnan(px)] = median_val if not np.isnan(median_val) else 0.0
        
    thermal_2d = px.reshape(sensor_shape)
    upscaled = cv2.resize(thermal_2d, target_shape, interpolation=cv2.INTER_CUBIC)
    return thermal_2d, upscaled


def plot_folder_verification(folder_name, base_dir=DATASET_DIR, num_samples=5, colormap='jet', random_seed=42):
    """
    Plots a 2-row comparison for random sample frames from a subfolder:
      Row 1: Joblib thermal pixel array upscaled to 320x240
      Row 2: Saved PNG image from the corresponding subfolder image directory
    """
    fpath = os.path.join(base_dir, folder_name)
    joblib_path = os.path.join(fpath, f"{folder_name}.joblib")
    img_dir = os.path.join(fpath, f"{folder_name}_image")
    
    if not os.path.exists(joblib_path):
        print(f"Error: Joblib file not found for {folder_name}")
        return
        
    jdata = joblib.load(joblib_path)
    total_frames = len(jdata)
    
    if total_frames == 0:
        print(f"No frames found in {folder_name}")
        return
        
    random.seed(random_seed)
    num_samples = min(num_samples, total_frames)
    sample_indices = random.sample(range(total_frames), num_samples)
    
    fig, axes = plt.subplots(2, num_samples, figsize=(4 * num_samples, 7))
    if num_samples == 1:
        axes = np.expand_dims(axes, axis=1)
        
    fig.suptitle(f"Verification Plot for Subfolder: {folder_name} ({total_frames} Total Frames)", fontsize=14, fontweight='bold', y=0.98)
    
    for col_idx, idx in enumerate(sample_indices):
        item = jdata[idx]
        frame_num = item.get('frame_number', idx)
        timestamp = item.get('timestamp', 0)
        pixels = item.get('pixels', [])
        
        # Process joblib thermal pixels
        _, upscaled = process_thermal_pixels(pixels, target_shape=(320, 240))
        
        valid_px = np.array(pixels, dtype=float)
        min_t, max_t = np.nanmin(valid_px), np.nanmax(valid_px)
        
        # Row 1: Joblib upscaled plot
        im0 = axes[0, col_idx].imshow(upscaled, cmap=colormap)
        axes[0, col_idx].set_title(f"Row 1: Joblib (320x240 Upscaled)\nFrame: {frame_num}\nTemp: [{min_t:.1f}°C - {max_t:.1f}°C]", fontsize=9)
        axes[0, col_idx].axis('off')
        plt.colorbar(im0, ax=axes[0, col_idx], fraction=0.046, pad=0.04)
        
        # Row 2: Saved PNG image match
        ts_ms = int(timestamp * 1000)
        img_name = f"{folder_name}_{frame_num}_{ts_ms}.png"
        img_path = os.path.join(img_dir, img_name)
        
        # Fallback matching by frame number
        if not os.path.exists(img_path) and os.path.exists(img_dir):
            matches = glob.glob(os.path.join(img_dir, f"*{frame_num}*.png"))
            if matches:
                img_path = matches[0]
                
        if os.path.exists(img_path):
            img = Image.open(img_path)
            axes[1, col_idx].imshow(img)
            axes[1, col_idx].set_title(f"Row 2: Saved Image\n{os.path.basename(img_path)}", fontsize=8)
        else:
            axes[1, col_idx].text(0.5, 0.5, "Image Not Found", ha='center', va='center', color='red', fontsize=12)
            axes[1, col_idx].set_title(f"Row 2: Missing Image", fontsize=9)
        axes[1, col_idx].axis('off')
        
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    print("=" * 80)
    print("RUNNING DATASET INTEGRITY AUDIT")
    print("=" * 80)
    audit_dataset_integrity(DATASET_DIR)
    
    subfolders = [d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))]
    subfolders.sort()

    print(f"\nRunning verification plots for all {len(subfolders)} subfolders...\n")
    for folder in subfolders:
        print("=" * 80)
        print(f"ANALYZING SUBFOLDER: {folder}")
        print("=" * 80)
        plot_folder_verification(folder_name=folder, base_dir=DATASET_DIR, num_samples=5, colormap='jet')


Dataset Directory set to: E:\MLX90640\Abbas Dataset\dataset-IR
RUNNING DATASET INTEGRITY AUDIT

Subfolder Name                   | Joblib   | JSON     | Images   | Aligned?
---------------------------------------------------------------------------
Empty_Room_No_Person             | 1533     | 1533     | 1533     | True    
Single_Person_Lying_East         | 310      | 310      | 310      | True    
Single_Person_Lying_West         | 224      | 224      | 224      | True    
Single_Person_Sitting_Center     | 111      | 111      | 111      | True    
Single_Person_Sitting_East       | 215      | 215      | 215      | True    
Single_Person_Sitting_West       | 181      | 181      | 181      | True    
Single_Person_Standing_Center    | 93       | 93       | 93       | True    
Single_Person_Standing_East      | 210      | 210      | 210      | True    
Single_Person_Standing_West      | 231      | 231      | 231      | True    

Running verification plots for all 9 subfolders...

ANALY